# L1a: Values and Primitive Data Types

Every value in a computer program has a type. This lecture focuses on Julia's primitive values: how their types determine storage, interpretation, and valid operations.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Inspect a value's representation:__ Report the type, the storage width, and the bit pattern of any primitive value, and explain what each of those three answers tells you that the other two do not.
> * __Compare primitive types:__ Explain why the integer, Boolean, and floating-point families use different numbers of bytes, and why the same stored bit pattern means different numbers when read as different types.
> * __Use character types:__ Recognize a `Char` as one Unicode character, distinguish it from its numeric code point, and convert a character to that code point.

Let's get started!
___


## Setup, Data, and Prerequisites

Run the local setup cell first. It activates the single pinned course environment, loads every package used by this meeting, and includes any meeting source code.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [7]:
include(joinpath(@__DIR__, "Include.jl")); #  include the Include.jl file

LoadError: LoadError: failed to find source of parent package: "ArrayInterface"
in expression starting at /Users/williammanno/CHEME-5800-CourseRepository-Fall-2026/Include.jl:22
in expression starting at /Users/williammanno/CHEME-5800-CourseRepository-Fall-2026/weeks/week-01/L1a/Include.jl:36

The course environment also loads [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl); see [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/). This notebook does not need it; this notebook uses [only Julia's `Base` library](https://docs.julialang.org/en/v1/base/base/#Base). We start using the package later in the course.

___

## Primitive Data Types
Primitive data types are the basic building blocks a language provides. They are _atomic_: they are not made from other types, and they hold simple values such as numbers, characters, and truth values. The smallest unit of storage is a _byte_, which is 8 bits. A _bit_ is a binary digit, either 0 or 1. A _bit pattern_ is a sequence of bits, and the _width_ of a type is the number of bytes it uses.

> __Why do types matter?__
>
> For the primitive types in this lecture, the language sets how many bytes a value uses, how it reads the sequence of 0s and 1s in those bytes, and which operations the compiler (or interpreter) will allow. The same bits can mean entirely different numbers when read as different types.
>
> For example, the bit pattern `11111111` means `255` as a `UInt8`, but `-1` as an `Int8`.

We start with [Integers](https://docs.julialang.org/en/v1/base/numbers/#Core.Int) and [the `Bool` type](https://docs.julialang.org/en/v1/base/numbers/#Core.Bool), then turn to floating-point values and characters.

### Integer and Boolean Types
An __integer__ represents a whole number $x\in\mathbb{Z}$: positive, negative, or zero. Julia stores integers in a _fixed-width_ binary form, typically 32 or 64 bits. A __boolean__, which is a type that can only have the values `true` or `false`, represents a truth value and is stored as a single bit in a byte. 

The `Bool` type is a subtype of the integer family, so it can be used in arithmetic expressions, but it is not a number in the usual sense.
Let's bind a whole number to the variable `x::Int64`:

In [3]:
x = 2 |> Int64; # select a whole number ... -2, -1, 0, 1, 2, ...

Every Julia value has its own type, and [the `typeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Core.typeof) reports it:

In [4]:
typeof(x) # this returns the type of the argument

Int64

The type tells us how the bits are _interpreted_. [The `bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring) shows us how the bits themselves are stored and arranged as a fixed-width string of 0s and 1s:

> __Bitstring representation:__
>
> We can see the stored bit pattern of a primitive value with [the `bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring). It takes a primitive value and returns a string of 0s and 1s, one character per bit, most significant bit first. For example, an `Int64` produces a 64-character string.
> 
> The order is a fixed display rule: [the `bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring) always writes the most significant bit on the left, whatever the machine does internally. 
> 
> [The `bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring) works on primitive values like `Int64` and `Float64`. Passing a `Tuple` or a `String` throws [an `ArgumentError`](https://docs.julialang.org/en/v1/base/base/#Core.ArgumentError) because these are not primitive values.

So what is actually stored for `x`?

In [5]:
bitstring(x) # shows the bit pattern stored in memory

"0000000000000000000000000000000000000000000000000000000000000010"

The bit pattern has to use space. [The `sizeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.sizeof-Tuple%7BType%7D) reports how much, measured in __bytes__:

In [6]:
sizeof(x) # number of bytes used to store x::Int64

8

__Boolean Values:__

A variable of type `Bool` ranges over $\mathbb{B} = \left\{\text{true},\text{false}\right\}$, so it carries exactly one bit of information. Let's bind `false` to `flag::Bool` and look at its representation:

In [7]:
flag = false; # the flag variable can take on values of {true | false}

The pattern is the same as before. First the type:

In [8]:
typeof(flag)

Bool

Then the stored bits:

In [9]:
bitstring(flag) # this should be 8 bits wide

"00000000"

A `Bool` carries a single bit of information, but memory addresses refer to whole __bytes__, so the smallest unit is one byte (or 8 bits). Let's ask [the `sizeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.sizeof-Tuple%7BType%7D) how much space `flag` takes up, and compare it against the eight bytes we just measured for `x`:

In [10]:
sizeof(flag) # number of bytes used to store the Bool

1

___

### Floating-Point Types
Floating-point types model real numbers using three parts defined by [the IEEE 754 standard](https://en.wikipedia.org/wiki/IEEE_754): a sign bit, an exponent (the scale), and a significand (also called the mantissa). Only the _fractional part_ of the significand is stored in memory; the leading digit is implicit, and for normalized values it is a `1`. 

We take a floating-point number apart bit by bit in `L1c`, where that implicit leading digit matters.

> __Julia versus Python floating point numbers__: Julia provides three standard IEEE-754 floating-point types that trade off precision for storage: `Float16` (half-precision), `Float32` (single-precision), and `Float64` (double-precision). 
> 
> __Python's built-in `float` type__: Python's built-in `float` type is normally a C `double` (IEEE-754 binary64 on mainstream systems), while numerical libraries such as NumPy and PyTorch provide fixed-width types such as `float16`, `float32`, and `float64` for arrays and tensors.

Let's look at a few examples. First, here's a 64-bit number (Julia's default):

In [ ]:
let
    x = 54.13; # default: in Julia, the default floating point number is 64-bit.
    bitstring(x)
end

"0100000001001011000100001010001111010111000010100011110101110001"

The same decimal literal, converted to 32 bits, has a different memory layout. It is also no longer exactly the same number: `Float32` cannot represent `54.13` as closely as `Float64` can (the precision story that we explore in `L1c`):

In [12]:
let
    x = 54.13 |> Float32 # cast to Float32 (single precision), not Float64
    bitstring(x) # gives a string with the bit pattern
end

"01000010010110001000010100011111"

Fewer bits means less storage and, as we will see in `L1c`, less precision. `Float16` halves the width again:

In [13]:
let
    x = 54.13 |> Float16 # cast to Float16 (half precision), not Float64
    sizeof(x) # returns number of bytes used to store x
end

2

___

### Character Types
Text on computers is made of characters, and each character has a unique integer called its __code point__. Traditional systems used [ASCII](https://en.wikipedia.org/wiki/ASCII) with one byte per character, while modern systems use [Unicode encodings like UTF-8 or UTF-16](https://en.wikipedia.org/wiki/Unicode) to represent a much wider range of characters.

> __Characters are not integers:__
>
> [The `Char` type](https://docs.julialang.org/en/v1/base/strings/#Core.Char) represents one Unicode character and is written using single quotes. It is not an integer type (`Char <: Integer` is `false`), but every character has a numeric Unicode __code point__.

Let's create [a `Char` in Julia](https://docs.julialang.org/en/v1/manual/unicode-input/) (notice the single quotes):

In [14]:
c = '🍣' # example Unicode character in Julia. See: https://docs.julialang.org/en/v1/manual/unicode-input/

'🍣': Unicode U+1F363 (category So: Symbol, other)

What is the code point (the unique integer) for the character `c`? We convert it with [the `UInt32(...)` constructor](https://docs.julialang.org/en/v1/base/numbers/#Core.UInt32):

In [15]:
code = UInt32(c) # convert the code point to an unsigned integer

0x0001f363

_Hmmm, what?_ This strange-looking result is a hexadecimal (base-16) representation of a 32-bit unsigned integer. The `code` variable is an ordinary 32-bit unsigned integer; Julia simply __displays__ unsigned integers in [hexadecimal](https://en.wikipedia.org/wiki/Hexadecimal), or base 16, and the `0x` prefix marks this format. 

The same value written in base 10 is `127843`. We study representations in other bases in `L1c`.

[The `Char` type](https://docs.julialang.org/en/v1/base/strings/#Core.Char) is a primitive type that represents one character, not a collection of bytes. In `L1b`, we will study collection types, which group several values together.

___

## Looking ahead

Primitive values become useful when we organize them. In Lab `L1b`, we will choose among tuples, arrays, sets, and dictionaries, then build a custom composite type. Lecture `L1c` returns to the floating-point bit pattern and takes it apart field by field.

___


## Summary

Every Julia value has a type that sets how it is stored in memory and which operations are valid on it.

> __Key Takeaways:__
>
> * **Bits alone carry no meaning:** A stored pattern becomes a number only when a type sets its width and tells Julia how to read it. This is why one bit pattern can mean an integer under one type and a floating-point value under another.
> * **Width is a decision, not a detail:** Wider primitive types use more bytes to provide more range or precision. Choosing `Float32` instead of `Float64`, or `Int32` instead of `Int64`, trades memory for range or precision.
> * **A character is not its code point:** A `Char` represents one Unicode character, while its code point is the integer Unicode assigns to that character. Julia can convert between them, but they are different types.

Every representation question later in the course asks the same things at a larger scale: what is stored, how wide is it, and how is it read? The answers become more complex, but the three questions do not change.
___